<a href="https://colab.research.google.com/github/dom-dang/7.C01/blob/main/7_C01_PSET_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  <center> Problem Set 6 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51<center>
<center> Due: Monday, May 11, 2026 at 3:00 PM ET. <center>



<b>Name:</b> Dominique Dang

<b>Kerberos id:</b> ddang

### Learning Objective
The objective of this problem set is to learn how to leverage foundation models, specifically Protein Language Models (pLMs) like ESM2, for downstream predictive tasks. You will generate and analyze protein embeddings, train and evaluate a Multi-Layer Perceptron (MLP) for protein function prediction using Gene Ontology (GO) terms, and compare this against an MLP trained on naive one-hot encoded sequences. Finally, you will use ESMFold to predict a protein's 3D structure from its sequence.


### Instructions
- This problem set has two modeling tasks with several sub-questions. Some are marked grad version, which are required for graduate students (X.C51) but optional for others. Points for all students are in <font color="blue">blue</font>, while grad-only points are in <font color="orange">orange</font>. There is one problem that is undergrad only in <font color="purple">purple</font>. The total points are 75 for undergraduates and 100 for graduates.

- To get started, make your own copy of this notebook template in Colab (e.g., "Save a copy in Drive") before editing.

    - Important: this problem set requires a GPU. In Google Colab go to `Edit -> Notebook settings` and set the `Hardware accelerator` to a GPU before running the notebook (changing the runtime resets the notebook). See the GPU section below for additional help.

- Collaboration is encouraged and AI tools are permitted, but submitting work that is not your own is plagiarism. Any collaboration or assistance from others or from an LLM (including utilities integrated in Colab) must be described at the end of your submission.

- Additional notes about how to use this template:
    - Put your code in the code blocks flagged with `############# Code ##########`.

    -  Numerical answers yielded from running the code should be included in an Answer Block (see next cell).

    - We have provided print statements where numerical answers are expected.

    -  Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.

    - When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

- Submission: upload your completed `pset1.ipynb` to Gradescope. Ensure the notebook runs without error and includes all necessary code, plots, and outputs. Comments are encouraged; place conceptual answers in Markdown/Text cells.


### Background (optional)
Protein Language Models (pLMs), including Meta's [ESMFold](https://www.science.org/doi/10.1126/science.ade2574), are used to generate protein structures given an input sequence. ESMFold stems from the ESM2 model, which generates embeddings for protein sequences, leveraging the transformer architecture (similar to the Bidirectional Encoder Representations from Transformers (BERT) model in the context of Natural Language Processing (NLP)).

ESM2 was trained using a masked language modeling objective, where a given position in the input amino acid sequence is masked and the model has to predict the most likely amino acid at that position. This is repeated for every amino acid in the sequence. The transformer architecture helps capture long-range relationships between different parts of a protein sequence without explicitly providing any information about the protein's 3D structure. ESM2 was trained on approximately 65 million sequences from the [UniRef](https://www.uniprot.org/help/uniref) protein sequence database. The ESM2 model you will be working with in this PSET has around 3 billion parameters, 36 layers, 40 attention heads, and a hidden space dimension of 2560.

In part 1, you will leverage a pre-trained ESM2 model to generate embeddings for protein sequences and use them to train a small MLP to predict GO (Gene Ontology) terms for proteins. The GO knowledge base includes subontologies describing molecular functions of proteins, biological processes in which proteins are involved, and cellular components in which they are active. As expected, GO annotations can generally be propagated to homologous proteins. The [UniProtKB/Swiss-Prot database](https://www.uniprot.org/help/uniprotkb_sections) contains manually curated GO annotations for thousands of organisms and more than 550,000 proteins. You can learn more about GO terms [here](https://geneontology.org/).

Here, you will compare the performance of the MLP-ESM2 model with an MLP trained on one-hot encoded sequence data. The first task has been accomplished in recent publication [author et al.](#references). You can skim these papers if you would like, but that is not required to answer the questions in the PSET. All the required background is explained below.

In part 3, you will use ESMFold to predict the structure of one protein in the validation set.


# Install required packages & Mount Google Drive (0 points)

In [ ]:
!pip install fair-esm scikit-learn biopython matplotlib torch pandas tqdm transformers

In [ ]:
pip install torch transformers scikit-learn numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
from Bio import SeqIO
import seaborn as sns
import csv
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import umap
import matplotlib.pyplot as plt
import pickle
import os
from google.colab import drive
from collections import Counter
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, precision_recall_curve
from tqdm import tqdm
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Get Training & Validation Data (0 points)

Run the cells below to extract the training and validation data.

In [ ]:
drive_path = '/content/drive/MyDrive/PSETS_ML4MolEng_2025/ps6/data/mf/'  # Change this to your path

In [ ]:
import pandas as pd
train_data_path = os.path.join(drive_path, 'train_data.pkl')
sampled_train_data = pd.read_pickle(train_data_path)
valid_data_path = os.path.join(drive_path, 'valid_data.pkl')
sampled_valid_data = pd.read_pickle(valid_data_path)

### Part 1: Leverage foundation models to predict protein structure and function
This part will require you to implement a deep neural network in PyTorch. If you need to, please go through the PyTorch tutorial [here](https://pytorch.org/tutorials/beginner/basics/intro.html) and the quickstart guide [here](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html). You'll want a GPU for this part - request one now.


**Task 1.1**: <font color="blue">(15 points)</font> Generate and Visualize ESM2 Embeddings

The output of layers in ESM models — so-called hidden states — can be converted into vector embeddings for amino acid sequences. These hidden states can be passed through a language modeling head to produce logits at each position, which are then transformed into probabilities using a softmax function. These probabilities can provide interpretable insights into the model’s understanding of protein sequences. Extract the ESM2 embeddings for both the training and validation datasets.



Project the ESM2 embeddings to 2 dimensions and visualize them using UMAP. <font color="blue">**(2.5 points)**</font>


-  What do you observe in terms of the information captured by the ESM2 embeddings for the training vs. validation sets? <font color="blue">**(2.5 points)**</font>

-  How might the observed patterns in the distribution of training vs. validation data affect out-of-distribution testing and generalization performance? <font color="blue">**(5 points)**</font>

-  What information about proteins can be useful when doing a train-test-validation split on sequence data? <font color="blue">**(5 points)**</font>


**Write answer here.**


# 1.1 Generate and Visualize ESM2 Embeddings (15 points)



Project the ESM2 embeddings to 2 dimensions and visualize them using UMAP. (2.5 points)


* What do you observe in terms of the information captured by the ESM2 embeddings for the training vs. validation sets? (2.5 points)

* How might the observed patterns in the distribution of training vs. validation data affect out-of-distribution testing and generalization performance? (5 points)

* What information about proteins can be useful when doing a train-test-validation split on protein sequences? (5 points)

In [ ]:
## ANSWER ##
# ESM2 embeddings for the training vs. validation sets #


In [ ]:
##ANSWER##
# Plot observed patterns in the distribution of training vs. validation data #

Answer:


**Task 1.2**: <font color="blue">(5 points)</font> Extract and binarize GO terms for training and validation data

Binarize the labels using `MultiLabelBinarizer`, and print the total number of unique GO terms in the dataset. Then, complete the preprocessing of the GO terms for the validation data using the `MultiLabelBinarizer.transform` function.


# 1.2 Extract and binarize GO annotations for training and validation data (5 points)

Run the cells below to extract the GO terms (labels) for the training data, binarize the labels using `MultiLabelBinarizer`, and print the total number of unique GO terms in the dataset. Then, complete the preprocessing of the GO terms for the validation data using the `MultiLabelBinarizer.transform` function.

In [ ]:
if isinstance(sampled_train_data['annotations'].iloc[0], str):
    sampled_train_data['annotations'] = sampled_train_data['annotations'].apply(ast.literal_eval)

all_go_terms = sorted(set(term for go_list in sampled_train_data['annotations'] for term in go_list))
print(f"Number of unique GO terms is {len(all_go_terms)}")

mlb = MultiLabelBinarizer(classes=all_go_terms)
train_go_annotations = mlb.fit_transform(sampled_train_data['annotations'])
sampled_train_data['go_annotations'] = list(train_go_annotations)



# Filter out GO terms that are not in mlb class defined above for training set so train and validation set share labels
if isinstance(sampled_valid_data['annotations'].iloc[0], str):
    sampled_valid_data['annotations'] = sampled_valid_data['annotations'].apply(ast.literal_eval)

known_go_terms = set(mlb.classes_)

sampled_valid_data['filtered_annotations'] = sampled_valid_data['annotations'].apply(
    lambda go_list: [term for term in go_list if term in known_go_terms])


#TODO#

In [ ]:
##ANSWER##


**Task 1.3**: <font color="blue">(5 points)</font> Creating datasets for model training and validation

Complete the definition of the `ProteinDataset` class shown below. Both the ESM2 embeddings and labels should be converted to `torch.float32`. **(2.5 points)** Then, create datasets and dataloaders for the training and validation data using the `ProteinDataset` class you defined. Don't forget to shuffle the training dataloader but not the validation dataloader. **(2.5 points)**


# 1.3 Creating datasets for model training and validation (5 points)

Complete the definition of the `ProteinDataset` class shown below. Both the ESM2 embeddings and labels should be converted to torch.float32 (2.5 points). Then create datasets and dataloaders for the training and validation data using the `ProteinDataset` class you defined. Don't forget to shuffle the training dataloader but not the validation dataloader (2.5 points).

In [ ]:
batch_size=32

class ProteinDataset(Dataset):
    def __init__(self, dataframe, go_annotations):

        ## TODO ##

    def __len__(self):
      return len(self.embeddings)

    def __getitem__(self, idx):
      self.embeddings[idx], self.labels[idx]

In [ ]:
## TODO ##
train_dataset =
valid_dataset =

train_loader =
valid_loader =

**Task 1.4**: <font color="blue">(5 points)</font> Define the MLP

Complete the definition of the `ESM2MLP` class with the right dimensions.


# 1.4 Define MLP (5 points)

Complete the definition of the ESM2MLP class with the right dimensions. (5 points)

In [ ]:
class MLPBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # First block
        if not hasattr(self, 'is_residual'):
            return F.relu(self.layer_norm(self.dropout(self.dense(x))))

        # Second block with residual connection
        identity = x
        x = self.dense(x)
        x = self.layer_norm(x)
        x = F.relu(x)
        return identity + x \


class ESM2MLP(nn.Module):
    def __init__(self, num_classes=?):  #TODO
        super().__init__()
        self.blocks = nn.Sequential(
            MLPBlock(?, 1024),  #TODO
            MLPBlock(1024, 1024),
            nn.Linear(1024, num_classes))
        self.blocks[1].is_residual = True

    def forward(self, x):
        return self.blocks(x)


#Instantiate model
model = ESM2MLP()

**Task 1.5**: <font color="blue">(30 points)</font> Train and Evaluate Model

To train the model, start by setting up the Adam optimizer, with a learning rate `lr` of `1e-4`. (**5 points**) Next, write your training loop using the provided `train_epoch()` function, and train the model for 10 epochs. This is a classification problem, so you will want to compute the binary cross-entropy loss using `nn.BCEWithLogitsLoss()`, which requires you to input classification probabilities and the ground truth labels from your data. (**10 points**) Record the loss across epochs and plot the training loss across epochs. Comment briefly on the trend you observe. (**5 points**)



To evaluate the model, run the provided code cells and answer the following questions:


- Why do we need the sigmoid activation when looking at the predictions and converting them to probabilities? (**5 points**)

- Compute Precision, Recall, and F1-Score using the relevant scikit-learn function and comment on the values. Note the `'average'` argument. (**5 points**)


**Write answer here.**


# 1.5.1 Train Model (20 points)

To train the model, start by setting up the Adam optimizer, with a learning rate `lr` of `1e-4`. (5 points)

Next, write your training loop using the provided `train_epoch()` function, and train the model for 10 epochs. This is a classification problem, so you will want to compute the binary cross-entropy loss using `nn.BCEWithLogitsLoss()`, which requires you to input classification probabilities and the ground truth labels from your data. (10 points)


Record the loss across epochs and plot the training loss across epochs. Comment briefly on the trend you observe. (5 points)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in tqdm(loader, desc="Training"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
##TODO##

In [ ]:
#TODO#

ANSWER:

# 1.5.2 Evaluate Model (10 points)

To evaluate the model, run the code block below.

Why do we need the sigmoid activation when looking at the predictions and converting them to probabilities? (5 points)

Compute Precision, Recall, and F1-Score using the relevant scikit-learn function and comment on the values. Note the 'average' argument. (5 points)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(valid_dataset.embeddings.to(device))
    probabilities = torch.sigmoid(logits).cpu().numpy()

# Binary predictions with 0.5 threshold
binary_predictions = (probabilities >= 0.5).astype(np.float32)

# Get top 5 predicted indices per sample
top_hits = np.argsort(-probabilities, axis=1)[:, :5]

# Ensure top-5 predictions are always marked as 1
for i, indices in enumerate(top_hits):
    binary_predictions[i, indices] = 1

# Convert indices to GO term strings
top_go_terms = [[mlb.classes_[idx] for idx in row] for row in top_hits]
sampled_valid_data['top_5_predicted_go_terms'] = top_go_terms

In [ ]:
##TODO##

**Task 1.6**: <font style="blue">(30 points)</font> Compare with MLP trained on one-hot encoded sequences


# 1.6 Compare with MLP trained on one-hot encoded sequences (30 points)




We will compare the performance of an **MLP trained on ESM2 embeddings** with a simpler **MLP trained on one-hot encoded sequence data** using the validation data.

Start by one-hot encoding the sequence data using the `one_hot_encode_sequence` function that you will have to complete (5 points). Then, complete the `ProteinOneHotDataset` class and create the training and validation dataloaders (5 points). Make sure to use the `collate_fn` to handle variable-length sequences when defining the DataLoader.

Implement the small MLP with the following architecture (5 points):

To do so, define the `__init__` function of the `OneHotMLP` class with `hidden_dim = 256`. You will also need to define `input_dim` and `num_classes`.

The class should inherit from `nn.Module`, and use these layers to build the network:
- `nn.Linear`
- `nn.ReLU`
- `nn.Dropout`
- `nn.Sequential`

The feedforward neural network should:
1. Take an input of shape `(batch_size, input_dim)`.
2. Have a hidden layer with **256** nodes and ReLU activation.
3. Map the output to **num_classes** labels.
4. Apply a **0.1** dropout rate after ReLU.

The `forward` method is already implemented for you.

Then, train the model using the same hyperparameters as defined for the MLP trained on ESM2 embeddings (**5 points**).

Evaluate the model on the validation data. Use a threshold of 0.5 to compute the binary predictions (**5 points**).

Compute Precision, Recall, and F1-Score as you did above. Comment on the values and compare performances of the MLP trained on one-hot encodings vs. MLP trained on ESM2 embeddings, and explain potential reasons for differences in performance (**2.5 points**). In your answer, make sure to comment on potential issues with one-hot encodings as well as on the information content of ESM2 embeddings (**2.5 points**).

In [ ]:
#One-hot encode sequences
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}

def one_hot_encode_sequence(seq, aa_to_int):

      ##TODO##

    return one_hot

##TODO##

In [ ]:
#Define Dataset Class and Dataloders

batch_size=8

class ProteinOneHotDataset(Dataset):
    def __init__(self, dataframe, go_annotations):
        ##TODO##

    def __len__(self):
        ##TODO##

    def __getitem__(self, idx):
        ##TODO##
        return sequence, label


def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences = pad_sequence(sequences, batch_first=True)  # Pad sequences to same length
    labels = torch.stack(labels)
    return sequences, labels


##TODO##

train_dataset =
valid_dataset =
train_loader =
valid_loader =

In [ ]:
#Define and call MLP


class OneHotMLP(nn.Module):

    ##TODO##

    def forward(self, x):
        x = x.mean(dim=1)  #mean-pooling across the sequence length dimension
        return self.network(x) #pass the averaged sequence through the model

model = OneHotMLP()

Train the model using the same hyperparameters as defined above (5 points).


In [ ]:
#Train model

##TODO##

In [ ]:
#just for debugging purposes - students dont have to plot training loss for this question
plt.plot((range(0,num_epochs)), training_losses, color='b')
plt.xlabel("Epochs")
plt.ylabel("Training Loss")
plt.title("Training Loss Across Epochs")

Evaluate the model on the validation data. Use a threshold of 0.5 to compute the binary predictions. (5 points)

In [ ]:
##ANSWER##

In [ ]:
##ANSWER##

#Evaluate model


In [ ]:
##TODO##

In [ ]:
#ANSWER#
# Print precision, recall and f1


ANSWER:

**Task 2**: Investigating a protein of interest: Mouse hypothalamic galanin-like neuropeptide (`GALP_MOUSE`) <font color="blue">(5 points)</font>

Compare the GO terms predicted by the MLP and the naive approach for `GALP_MOUSE` in the validation set and its experimental annotations. Note that you can ignore the '`|IEA, `' and '`|IDA`' suffixes in the GO terms. These relate to how the annotations were made - more details can be found [here](http://geneontology.org/GO_REF/0000028.html). Use this [link](https://www.ebi.ac.uk/QuickGO/) to look up the GO terms.


# 2. Investigating a protein of interest: Mouse hypothalamic galanin-like neuropeptide (GALP_MOUSE) (5 points)

Compare the GO terms predicted by the MLP and naive approach for `GALP_MOUSE` in the validation set and its experimental annotations.

Note: you can ignore '|IEA, '|IDA' suffixes in the GO terms. These relate to how the annotations were made - more [here](http://geneontology.org/GO_REF/0000028.html).

Use this [link](https://www.ebi.ac.uk/QuickGO/) to look up the GO terms.

In [ ]:
#TODO#

In [ ]:
## ANSWER ##
#MLP trained on one hot encoded sequences


In [ ]:
## ANSWER ##
#MLP trained on ESM2 embeddings


**Task 3**: Predicting Protein Structure using ESMFold <font style="blue">(5 points)</font>

# 3. Predicting Protein Structure using ESMFold (5 points)



Use the relevant sequence from the validation dataframe to predict the 3D structure of `GALP_MOUSE` using [ESMFold](https://esmatlas.com/resources?action=fold).  

Paste the relevant protein sequence and hit **"Fold Sequence"**.  

Please upload below a screenshot of the top predicted structure using the **Files** tab to the left.  

Include the sequence you pasted into ESMFold in your answer below and comment on the confidence of the top prediction.  

In [ ]:
#TODO#

In [ ]:
##ANSWER##
